# Model Merging - Continue Interrupted Process

This notebook continues the model merging process that was interrupted during training.

It will:
1. Load the base model (Qwen3-14B-unsloth-bnb-4bit)
2. Load the trained LoRA adapter from `./model_output/Adapter/`
3. Merge them together
4. Save the merged model in 16-bit format to `./model_output/merged_16bit/`


## Setup Environment Variables (Windows Compatibility)


In [1]:
!pip show torch

Name: torch
Version: 2.9.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: f:\cursorprojects\weclone1020\.venv\lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, sympy, typing-extensions
Required-by: accelerate, bitsandbytes, cut-cross-entropy, peft, spacy-huggingface-pipelines, torchaudio, torchvision, unsloth, unsloth_zoo, xformers


In [2]:
import os
import sys

if sys.platform == "win32":
    os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

print("Environment variables set for Windows compatibility")


Environment variables set for Windows compatibility


## Import Libraries


In [2]:
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


f:\cursorprojects\WeClone1020\.venv\lib\site-packages\triton\knobs.py:212: UserWarning: Failed to find cuobjdump.exe
  warnings.warn(f"Failed to find {binary}")
f:\cursorprojects\WeClone1020\.venv\lib\site-packages\triton\knobs.py:212: UserWarning: Failed to find nvdisasm.exe
  warnings.warn(f"Failed to find {binary}")
[xformers|WARNING]WARNING[XFORMERS]: Need to compile C++ extensions to use all xFormers features.
    Please install xformers properly (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
Need to compile C++ extensions to use all xFormers features.
    Please install xformers properly (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.9.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5090 Laptop GPU


## Configuration


In [3]:
BASE_MODEL = "unsloth/Qwen3-14B-unsloth-bnb-4bit"
ADAPTER_PATH = "./model_output/Adapter"
OUTPUT_DIR = "./model_output/merged_16bit"
MAX_SEQ_LENGTH = 256

print(f"Base model: {BASE_MODEL}")
print(f"Adapter path: {ADAPTER_PATH}")
print(f"Output directory: {OUTPUT_DIR}")


Base model: unsloth/Qwen3-14B-unsloth-bnb-4bit
Adapter path: ./model_output/Adapter
Output directory: ./model_output/merged_16bit


## Step 1: Load Base Model


In [4]:
print("Loading base model...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    trust_remote_code=True
)

print(f"✓ Base model loaded successfully")
print(f"Model dtype: {model.dtype}")
print(f"Model device: {model.device}")


Loading base model...


f:\cursorprojects\WeClone1020\.venv\lib\site-packages\unsloth_zoo\gradient_checkpointing.py:348: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  GPU_BUFFERS = tuple([torch.empty(2*256*2048, dtype = dtype, device = f"{DEVICE_TYPE_TORCH}:{i}") for i in range(n_gpus)])


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.10.12: Fast Qwen3 patching. Transformers: 4.53.2.
   \\   /|    NVIDIA GeForce RTX 5090 Laptop GPU. Num GPUs = 1. Max memory: 23.889 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Base model loaded successfully
Model dtype: torch.bfloat16
Model device: cuda:0


## Step 2: Load LoRA Adapter


In [7]:
print(f"Loading LoRA adapter from {ADAPTER_PATH}...")

from peft import PeftModel

model = PeftModel.from_pretrained(model, ADAPTER_PATH)

print(f"✓ LoRA adapter loaded successfully")
print(f"Adapter config: {model.peft_config}")


Loading LoRA adapter from ./model_output/Adapter...
✓ LoRA adapter loaded successfully
Adapter config: {'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping={'base_model_class': 'Qwen3ForCausalLM', 'parent_library': 'transformers.models.qwen3.modeling_qwen3', 'unsloth_fixed': True}, base_model_name_or_path='unsloth/Qwen3-14B-unsloth-bnb-4bit', revision=None, inference_mode=True, r=32, target_modules={'o_proj', 'down_proj', 'q_proj', 'v_proj', 'up_proj', 'k_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=True, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeC

## Step 3: Check Memory Usage


In [5]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    current_memory = round(torch.cuda.memory_allocated() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    
    print(f"GPU: {gpu_stats.name}")
    print(f"Current memory usage: {current_memory} GB")
    print(f"Max memory: {max_memory} GB")
    print(f"Memory available: {max_memory - current_memory} GB")


GPU: NVIDIA GeForce RTX 5090 Laptop GPU
Current memory usage: 10.414 GB
Max memory: 23.889 GB
Memory available: 13.475 GB


## Step 4: Merge and Save Model (16-bit)

This will merge the LoRA weights with the base model and save as 16-bit model. This may take several minutes.


In [8]:
print(f"Merging LoRA adapter with base model and saving to {OUTPUT_DIR}...")
print("This may take several minutes depending on model size...")

try:
    model.save_pretrained_merged(
        OUTPUT_DIR,
        tokenizer,
        save_method="merged_16bit"
    )
    
    print("\n✓ Model merged and saved successfully!")
    print(f"Merged model saved to: {OUTPUT_DIR}")
    
except Exception as e:
    print(f"\n✗ Error during merging: {e}")
    print("\nTrying alternative method...")
    
    model = model.merge_and_unload()
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    
    print("\n✓ Model merged using alternative method!")
    print(f"Merged model saved to: {OUTPUT_DIR}")


Merging LoRA adapter with base model and saving to ./model_output/merged_16bit...
This may take several minutes depending on model size...

✓ Model merged and saved successfully!
Merged model saved to: ./model_output/merged_16bit


## Step 5: Verify Saved Model


In [ ]:
import os

print("Checking saved files...")
if os.path.exists(OUTPUT_DIR):
    files = os.listdir(OUTPUT_DIR)
    print(f"\n✓ Output directory exists with {len(files)} files:")
    for f in sorted(files):
        file_path = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path) / (1024**3)
            print(f"  - {f} ({size:.2f} GB)")
        else:
            print(f"  - {f} (directory)")
else:
    print(f"\n✗ Output directory not found: {OUTPUT_DIR}")


## Step 6: Test Merged Model (Optional)


In [ ]:
print("Loading merged model for testing...")

test_model, test_tokenizer = FastLanguageModel.from_pretrained(
    model_name=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
    trust_remote_code=True
)

print("✓ Merged model loaded successfully!")
print(f"Model dtype: {test_model.dtype}")
print(f"Model device: {test_model.device}")


In [ ]:
FastLanguageModel.for_inference(test_model)

test_messages = [
    {"role": "user", "content": "你好，你是谁？"}
]

inputs = test_tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(test_model.device)

print("Generating response...")
outputs = test_model.generate(
    input_ids=inputs,
    max_new_tokens=64,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

response = test_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nTest Response:\n{response}")


## Summary

The model merging process is complete! The merged 16-bit model is saved in:
```
./model_output/merged_16bit/
```

You can now use this merged model for inference or further fine-tuning.

### Next Steps:
1. Test the model with `weclone-cli test-model` (update settings.jsonc to point to merged model)
2. Start the web demo with `weclone-cli webchat-demo`
3. Deploy the model for production use

### File Structure:
- `./model_output/Adapter/` - Original LoRA adapter (keep for future use)
- `./model_output/merged_16bit/` - Merged 16-bit model (ready for inference)
